In [4]:

import os
import joblib
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# Device
device = torch.device("cpu")

# Load dataset
data = joblib.load("Dataset/cycle_time_data.joblib")

Xraw = np.asarray(data["Xraw"], dtype=np.float32)
yraw = np.asarray(data["yraw"], dtype=np.float32).reshape(-1, 1)

train_end = data["train_end"]
window = data["window"]
stable = data["stable"]
drift = data["drift"]
recovery = data["recovery"]

# Scale data using training observations only
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit(Xraw[:train_end]).transform(Xraw)
y_scaled = scaler_y.fit(yraw[:train_end]).transform(yraw)

# Define the same iTransformer architecture
class iTransformer(nn.Module):
    def __init__(self, seq_len=10, num_features=5):
        super().__init__()

        self.embedding = nn.Linear(seq_len, 32)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=32,
            nhead=4,
            dim_feedforward=64,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2,
            enable_nested_tensor=False
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_features * 32, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.embedding(x)
        x = self.encoder(x)
        return self.head(x).squeeze(-1)


# Create model
model = iTransformer(
    seq_len=window,
    num_features=X_scaled.shape[1]
).to(device)

# Load previously trained weights
model_path = "Results/iTransformer/itransformer_model.pth"

model.load_state_dict(
    torch.load(
        model_path,
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("Model loaded successfully!")
print("Model:", type(model).__name__)
print("Device:", device)
print("Window size:", window)
print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)


Model loaded successfully!
Model: iTransformer
Device: cpu
Window size: 10
Total parameters: 27809


In [5]:

# Step 12.1: Initialize periodic iTransformer

import copy
import os
import torch
import numpy as np

# Copy the original trained model
periodic_model = copy.deepcopy(model).to(device)
periodic_model.eval()

# Same fine-tuning parameters as ADWIN
periodic_learning_rate = 0.0001
periodic_epochs = 10
periodic_window = 100
periodic_batch_size = 16

# Update every 100 observations
update_interval = 100

# Store results
periodic_predictions = []
periodic_actuals = []
periodic_errors = []
periodic_update_points = []

os.makedirs(
    "Results/iTransformer_Periodic",
    exist_ok=True
)

# Verify identical initial weights
same_weights = all(
    torch.equal(p1, p2)
    for p1, p2 in zip(
        model.parameters(),
        periodic_model.parameters()
    )
)

print("Periodic iTransformer initialized.")
print("Update interval:", update_interval)
print("Adaptation window:", periodic_window)
print("Fine-tuning epochs:", periodic_epochs)
print("Initial weights identical:", same_weights)


Periodic iTransformer initialized.
Update interval: 100
Adaptation window: 100
Fine-tuning epochs: 10
Initial weights identical: True


In [6]:

# Step 12.2: Periodic iTransformer forecasting

import time
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

def periodic_fine_tune(model, current_index):

    start = max(
        window,
        current_index - periodic_window + 1
    )

    X_recent = []
    y_recent = []

    for j in range(start, current_index + 1):
        X_recent.append(X_scaled[j-window:j])
        y_recent.append(y_scaled[j, 0])

    X_tensor = torch.tensor(
        np.array(X_recent),
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        np.array(y_recent),
        dtype=torch.float32
    )

    loader = DataLoader(
        TensorDataset(X_tensor, y_tensor),
        batch_size=periodic_batch_size,
        shuffle=True
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=periodic_learning_rate
    )

    criterion = nn.MSELoss()
    model.train()

    for epoch in range(periodic_epochs):
        for X_batch, y_batch in loader:

            optimizer.zero_grad()

            pred = model(X_batch.to(device))
            loss = criterion(
                pred, y_batch.to(device)
            )

            loss.backward()
            optimizer.step()

    model.eval()


# Reset model to original trained weights
periodic_model.load_state_dict(
    torch.load(
        model_path,
        map_location=device,
        weights_only=True
    )
)

periodic_model.eval()

periodic_predictions = []
periodic_actuals = []
periodic_errors = []
periodic_update_points = []

start_time = time.time()

for i in range(train_end, len(X_scaled)):

    # Use only previous observations
    X_window = X_scaled[i-window:i]

    X_tensor = torch.tensor(
        X_window,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    periodic_model.eval()

    with torch.no_grad():
        pred_scaled = periodic_model(
            X_tensor
        ).cpu().numpy().reshape(-1, 1)

    pred = scaler_y.inverse_transform(
        pred_scaled
    )[0, 0]

    # Observe actual target after prediction
    actual = float(yraw[i, 0])
    error = abs(actual - pred)

    periodic_predictions.append(pred)
    periodic_actuals.append(actual)
    periodic_errors.append(error)

    # Update after every 100 test observations
    if (i - train_end + 1) % update_interval == 0:

        periodic_update_points.append(i)

        print(f"Periodic update at observation {i}")

        periodic_fine_tune(
            periodic_model,
            current_index=i
        )

execution_time = time.time() - start_time

print("\nPeriodic forecasting completed.")
print("Update points:", periodic_update_points)
print("Total updates:", len(periodic_update_points))
print(f"Execution time: {execution_time:.2f} seconds")

# Save predictions
results_df = pd.DataFrame({
    "Index": np.arange(train_end, len(X_scaled)),
    "Actual": periodic_actuals,
    "Predicted": periodic_predictions,
    "Absolute_Error": periodic_errors
})

results_df.to_csv(
    "Results/iTransformer_Periodic/periodic_predictions.csv",
    index=False
)

torch.save(
    periodic_model.state_dict(),
    "Results/iTransformer_Periodic/periodic_model.pth"
)

print("Periodic results saved successfully.")


Periodic update at observation 499
Periodic update at observation 599
Periodic update at observation 699
Periodic update at observation 799
Periodic update at observation 899

Periodic forecasting completed.
Update points: [499, 599, 699, 799, 899]
Total updates: 5
Execution time: 8.06 seconds
Periodic results saved successfully.


In [7]:

# Step 12.3: Compare three iTransformer models

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load predictions
baseline = pd.read_csv(
    "Results/iTransformer/predictions.csv"
)

adwin = pd.read_csv(
    "Results/iTransformer_ADWIN/adaptive_predictions.csv"
)

periodic = pd.read_csv(
    "Results/iTransformer_Periodic/periodic_predictions.csv"
)

# Merge by observation index
comparison = (
    baseline[["Index", "Actual", "Predicted"]]
    .rename(columns={"Predicted": "Baseline"})
    .merge(
        adwin[["Index", "Predicted"]].rename(
            columns={"Predicted": "ADWIN"}
        ),
        on="Index",
        validate="one_to_one"
    )
    .merge(
        periodic[["Index", "Predicted"]].rename(
            columns={"Predicted": "Periodic"}
        ),
        on="Index",
        validate="one_to_one"
    )
)

assert len(comparison) == 595
assert comparison["Index"].is_unique

# Confirm all three use the same actual targets
for source in [adwin, periodic]:
    aligned = comparison[["Index", "Actual"]].merge(
        source[["Index", "Actual"]],
        on="Index",
        suffixes=("_Reference", "_Other")
    )
    assert np.allclose(
        aligned["Actual_Reference"],
        aligned["Actual_Other"]
    )

# Evaluation periods
periods = {
    "Stable": (400, 600),
    "Drift": (600, 800),
    "Recovery": (800, 995),
    "Overall": (400, 995)
}

rows = []

for period_name, (start, end) in periods.items():

    subset = comparison[
        (comparison["Index"] >= start) &
        (comparison["Index"] < end)
    ]

    for model_name in ["Baseline", "ADWIN", "Periodic"]:

        actual = subset["Actual"]
        predicted = subset[model_name]

        mae = mean_absolute_error(actual, predicted)
        rmse = np.sqrt(
            mean_squared_error(actual, predicted)
        )

        rows.append({
            "Period": period_name,
            "Model": model_name,
            "MAE": round(mae, 3),
            "RMSE": round(rmse, 3)
        })

results = pd.DataFrame(rows)

print("Three-Model Comparison:")
display(results)

# Save comparison
results.to_csv(
    "Results/iTransformer_Periodic/"
    "three_model_comparison.csv",
    index=False
)

print("Comparison saved successfully.")


Three-Model Comparison:


,Period,Model,MAE,RMSE
0,Stable,Baseline,26.743,32.507
1,Stable,ADWIN,26.743,32.507
2,Stable,Periodic,26.895,32.682
3,Drift,Baseline,59.961,72.121
4,Drift,ADWIN,52.155,61.676
5,Drift,Periodic,53.861,63.940
6,Recovery,Baseline,33.604,39.243
7,Recovery,ADWIN,31.062,36.777
8,Recovery,Periodic,32.327,39.678
9,Overall,Baseline,40.158,51.071


Comparison saved successfully.


In [8]:

# Step 13.1: Periodic update interval sensitivity

import copy
import time
import os
import pandas as pd
import numpy as np
import torch

intervals = [50, 100, 200]

all_results = []
all_predictions = []

os.makedirs(
    "Results/iTransformer_Periodic/Sensitivity",
    exist_ok=True
)

# Use the same initial trained model for every interval
initial_weights = copy.deepcopy(model.state_dict())

for interval in intervals:

    print(f"\nTesting update interval: {interval}")

    # Reset model for independent experiment
    periodic_model = copy.deepcopy(model).to(device)
    periodic_model.load_state_dict(initial_weights)
    periodic_model.eval()

    predictions = []
    actuals = []
    update_points = []

    start_time = time.time()

    for i in range(train_end, len(X_scaled)):

        # Previous observations only
        X_window = X_scaled[i-window:i]

        X_tensor = torch.tensor(
            X_window,
            dtype=torch.float32
        ).unsqueeze(0).to(device)

        with torch.no_grad():
            pred_scaled = periodic_model(
                X_tensor
            ).cpu().numpy().reshape(-1, 1)

        pred = scaler_y.inverse_transform(
            pred_scaled
        )[0, 0]

        actual = float(yraw[i, 0])

        predictions.append(pred)
        actuals.append(actual)

        # Update after actual value becomes available
        if (i - train_end + 1) % interval == 0:

            update_points.append(i)

            periodic_fine_tune(
                periodic_model,
                current_index=i
            )

    elapsed = time.time() - start_time

    # Save predictions
    pred_df = pd.DataFrame({
        "Index": np.arange(train_end, len(X_scaled)),
        "Actual": actuals,
        "Predicted": predictions,
        "Interval": interval
    })

    all_predictions.append(pred_df)

    # Calculate metrics for each period
    for period_name, (start, end) in {
        "Stable": (400, 600),
        "Drift": (600, 800),
        "Recovery": (800, 995),
        "Overall": (400, 995)
    }.items():

        subset = pred_df[
            (pred_df["Index"] >= start) &
            (pred_df["Index"] < end)
        ]

        errors = (
            subset["Actual"] - subset["Predicted"]
        )

        mae = np.mean(np.abs(errors))
        rmse = np.sqrt(np.mean(errors ** 2))

        all_results.append({
            "Interval": interval,
            "Period": period_name,
            "MAE": round(mae, 3),
            "RMSE": round(rmse, 3),
            "Updates": len(update_points),
            "Time_Seconds": round(elapsed, 2)
        })

    print("Updates:", len(update_points))
    print("Execution time:", round(elapsed, 2), "seconds")

# Combine and save
sensitivity_results = pd.DataFrame(all_results)

sensitivity_predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

sensitivity_results.to_csv(
    "Results/iTransformer_Periodic/Sensitivity/"
    "interval_results.csv",
    index=False
)

sensitivity_predictions.to_csv(
    "Results/iTransformer_Periodic/Sensitivity/"
    "interval_predictions.csv",
    index=False
)

print("\nSensitivity Analysis Results:")
display(sensitivity_results)

print("\nAll results saved successfully.")



Testing update interval: 50
Updates: 11
Execution time: 8.28 seconds

Testing update interval: 100
Updates: 5
Execution time: 4.47 seconds

Testing update interval: 200
Updates: 2
Execution time: 2.3 seconds

Sensitivity Analysis Results:


,Interval,Period,MAE,RMSE,Updates,Time_Seconds
0,50,Stable,27.326,32.918,11,8.28
1,50,Drift,50.836,59.536,11,8.28
2,50,Recovery,32.321,39.592,11,8.28
3,50,Overall,36.866,45.491,11,8.28
4,100,Stable,26.884,32.673,5,4.47
5,100,Drift,53.743,63.815,5,4.47
6,100,Recovery,32.224,39.818,5,4.47
7,100,Overall,37.662,47.406,5,4.47
8,200,Stable,26.743,32.507,2,2.30
9,200,Drift,61.557,74.052,2,2.30



All results saved successfully.


In [9]:

# Step 14.1: Feature and target leakage inspection

import numpy as np
import pandas as pd

print("DATASET INFORMATION")
print("Total observations:", len(df) if "df" in globals() else len(data["df"]))
print("Target shape:", yraw.shape)
print("Features:", data["features"])

df_check = data["df"].copy()

print("\nFEATURE CORRELATION WITH TARGET")

check_df = pd.DataFrame(
    Xraw,
    columns=data["features"]
)

check_df["Target_Cycle_Time"] = yraw.flatten()

correlations = (
    check_df.corr(numeric_only=True)["Target_Cycle_Time"]
    .drop("Target_Cycle_Time")
    .sort_values(ascending=False)
)

display(correlations.to_frame("Correlation"))

print("\nTARGET COMPARISON")

for column in [
    "Processing_Time",
    "PT_orig",
    "Upstream_Lag"
]:
    if column in df_check.columns:
        values = pd.to_numeric(
            df_check[column],
            errors="coerce"
        ).to_numpy()

        target = yraw.flatten()

        valid = np.isfinite(values) & np.isfinite(target)

        print(f"\n{column}")
        print(
            "Equal to target (%):",
            round(
                np.mean(
                    np.isclose(
                        values[valid],
                        target[valid]
                    )
                ) * 100,
                2
            ) if valid.any() else "N/A"
        )

        print(
            "Correlation:",
            round(
                np.corrcoef(
                    values[valid],
                    target[valid]
                )[0, 1],
                4
            ) if valid.sum() > 2 else "N/A"
        )

print("\nFIRST 10 OBSERVATIONS")
display(
    check_df.head(10)
)


DATASET INFORMATION
Total observations: 995
Target shape: (995, 1)
Features: ['Processing_Time', 'Energy_Consumption', 'Machine_Availability', 'Upstream_Lag', 'Station_ID']

FEATURE CORRELATION WITH TARGET


,Correlation
Processing_Time,1.000000
Upstream_Lag,0.216848
Machine_Availability,0.010668
Station_ID,-0.004546
Energy_Consumption,-0.021577



TARGET COMPARISON

Processing_Time
Equal to target (%): 100.0
Correlation: 1.0

PT_orig
Equal to target (%): 60.3
Correlation: 0.8804

Upstream_Lag
Equal to target (%): 1.51
Correlation: 0.2168

FIRST 10 OBSERVATIONS


,Processing_Time,Energy_Consumption,Machine_Availability,Upstream_Lag,Station_ID,Target_Cycle_Time
0,79.0,6.61,84.0,76.0,0.0,79.0
1,106.0,12.50,95.0,56.0,1.0,106.0
2,46.0,8.13,88.0,79.0,0.0,46.0
3,22.0,14.20,87.0,106.0,1.0,22.0
4,42.0,8.97,81.0,100.0,2.0,42.0
5,27.0,3.66,97.0,46.0,0.0,27.0
6,65.0,9.98,90.0,27.0,0.0,65.0
7,43.0,5.55,95.0,28.0,4.0,43.0
8,112.0,2.01,95.0,22.0,1.0,112.0
9,104.0,3.67,98.0,79.0,3.0,104.0


In [10]:

# Step 14.3: Chronological ordering check

df_check = data["df"].copy()

time_columns = [
    "Scheduled_Start",
    "Scheduled_End",
    "Actual_Start",
    "Actual_End"
]

for col in time_columns:
    df_check[col] = pd.to_datetime(
        df_check[col],
        errors="coerce"
    )

print("TIME COLUMN INFORMATION")

for col in time_columns:
    print(
        col,
        "Missing:", df_check[col].isna().sum(),
        "Monotonic:", df_check[col].is_monotonic_increasing
    )

# Check if historical jobs finish before
# the next job's scheduled start
previous_end = df_check["Actual_End"].shift(1)
next_start = df_check["Scheduled_Start"]

valid = previous_end.notna() & next_start.notna()

available = (
    previous_end[valid] <= next_start[valid]
)

print("\nHISTORICAL TARGET AVAILABILITY")
print("Comparable pairs:", valid.sum())
print(
    "Previous job completed before next scheduled start (%):",
    round(available.mean() * 100, 2)
    if valid.any() else "N/A"
)

print("\nFIRST 10 TIMESTAMPS")
display(
    df_check[
        ["Job_ID", "Scheduled_Start",
         "Actual_Start", "Actual_End"]
    ].head(10)
)


TIME COLUMN INFORMATION
Scheduled_Start Missing: 0 Monotonic: True
Scheduled_End Missing: 0 Monotonic: False
Actual_Start Missing: 128 Monotonic: False
Actual_End Missing: 128 Monotonic: False

HISTORICAL TARGET AVAILABILITY
Comparable pairs: 866
Previous job completed before next scheduled start (%): 0.0

FIRST 10 TIMESTAMPS


,Job_ID,Scheduled_Start,Actual_Start,Actual_End
0,J002,2023-03-18 08:10:00,2023-03-18 08:20:00,2023-03-18 09:39:00
1,J004,2023-03-18 08:30:00,2023-03-18 08:35:00,2023-03-18 10:21:00
2,J005,2023-03-18 08:40:00,2023-03-18 08:42:00,2023-03-18 09:28:00
3,J007,2023-03-18 09:00:00,2023-03-18 09:04:00,2023-03-18 09:26:00
4,J009,2023-03-18 09:20:00,2023-03-18 09:23:00,2023-03-18 10:05:00
5,J010,2023-03-18 09:30:00,2023-03-18 09:54:00,2023-03-18 10:21:00
6,J011,2023-03-18 09:40:00,2023-03-18 09:38:00,2023-03-18 10:43:00
7,J013,2023-03-18 10:00:00,2023-03-18 10:05:00,2023-03-18 10:48:00
8,J014,2023-03-18 10:10:00,2023-03-18 10:10:00,2023-03-18 12:02:00
9,J015,2023-03-18 10:20:00,2023-03-18 10:24:00,2023-03-18 12:08:00


In [11]:

# Step 14.4: Full forecasting-window availability audit

import numpy as np
import pandas as pd

df_audit = data["df"].copy()

df_audit["Scheduled_Start"] = pd.to_datetime(
    df_audit["Scheduled_Start"], errors="coerce"
)

df_audit["Actual_End"] = pd.to_datetime(
    df_audit["Actual_End"], errors="coerce"
)

records = []

for i in range(train_end, len(df_audit)):

    forecast_time = df_audit.loc[i, "Scheduled_Start"]

    history = df_audit.iloc[i-window:i]

    # Historical target is available only if
    # its job finished by forecast time
    available = (
        history["Actual_End"].notna()
        & (history["Actual_End"] <= forecast_time)
    )

    records.append({
        "Index": i,
        "Available_History": int(available.sum()),
        "Unavailable_History": int((~available).sum()),
        "All_10_Available": bool(available.all())
    })

availability_df = pd.DataFrame(records)

print("FULL WINDOW AVAILABILITY AUDIT")
print("Test observations:", len(availability_df))

print(
    "Windows with all 10 historical targets available:",
    availability_df["All_10_Available"].sum()
)

print(
    "Percentage fully available:",
    round(
        availability_df["All_10_Available"].mean() * 100,
        2
    )
)

print(
    "Average available historical observations:",
    round(availability_df["Available_History"].mean(), 2)
)

print("\nAvailability distribution:")
display(
    availability_df["Available_History"]
    .value_counts()
    .sort_index()
    .rename_axis("Available observations")
    .reset_index(name="Forecast windows")
)

print("\nFirst 10 results:")
display(availability_df.head(10))


FULL WINDOW AVAILABILITY AUDIT
Test observations: 595
Windows with all 10 historical targets available: 0
Percentage fully available: 0.0
Average available historical observations: 2.66

Availability distribution:


,Available observations,Forecast windows
0,0,26
1,1,81
2,2,160
3,3,180
4,4,104
5,5,39
6,6,4
7,7,1



First 10 results:


,Index,Available_History,Unavailable_History,All_10_Available
0,400,2,8,False
1,401,3,7,False
2,402,3,7,False
3,403,2,8,False
4,404,1,9,False
5,405,0,10,False
6,406,0,10,False
7,407,0,10,False
8,408,2,8,False
9,409,3,7,False
